# CrossMed 02 - Train models
Train any subset of the registered experiment matrix. Checkpoints are saved independently for later evaluation.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from crossmed.data import UltrasoundDataset
from crossmed.models import build_model
from crossmed.experiments import load_experiment_matrix, run_experiment_matrix
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MATRIX = PROJECT_ROOT / 'configs/experiment_matrix.yaml'
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Verify one transformed sample before training

In [ ]:
dataset = UltrasoundDataset(
    PROJECT_ROOT / 'data/manifests/busbra_train.csv',
    image_size=224, train=True, data_root=PROJECT_ROOT / 'data',
)
sample = dataset[0]
print('Tensor:', sample['image'].shape, 'Label:', sample['label'].item(), 'ID:', sample['sample_id'])

## Available experiments

In [ ]:
experiments = load_experiment_matrix(MATRIX)
pd.DataFrame({'experiment': experiments.keys(), 'configuration': experiments.values()})

## Select experiments
Begin with one model to verify the pipeline. Use `list(experiments)` only when ready to run the complete sequence.

In [ ]:
SELECTED = ['resnet18_finetuned']
# SELECTED = ['resnet18_frozen', 'resnet50_frozen', 'resnet101_frozen']
# SELECTED = ['vit_b16_frozen']
# SELECTED = list(experiments)

FRESH_TRAINING = False
CLEANUP_BETWEEN_MODELS = True

## Train
With `FRESH_TRAINING=False`, existing checkpoints are preserved and skipped. Set it to `True` only to replace selected checkpoints.

In [ ]:
training_status = run_experiment_matrix(
    MATRIX,
    selected=SELECTED,
    stage='train',
    skip_completed=not FRESH_TRAINING,
    cleanup_between_runs=CLEANUP_BETWEEN_MODELS,
)
training_status

Each completed run writes `best.pt` and `training_history.json` under its own `outputs/` directory. The live model is unloaded before the next selected model when cleanup is enabled.